# Import module

In [ ]:
# Import module
from urllib.request import urlopen
import json
import re
from itertools import chain
import spacy

# Load data from original content database

In [ ]:
# Load full data from json file
with open("content_database.json", "r") as fi:
    data = json.load(fi)

data = data['response']['docs']

# Group by domain into new_data
new_data = {"medline": [], "webmd": [], "medscape": []} 
for item in data:
    if ("content" in item.keys()) & ("title" in item.keys()):
        # Select title, url, content
        new_dict = {}
        for key, value in item.items():
            if key in ["title", "url", "content"]:
                new_dict[key] = value[0]

        # Grouping
        for domain in new_data.keys():
            if domain in new_dict["url"]:
                new_data[domain].append(new_dict)
    

In [3]:
# Load SciSpaCy's medical model
nlp = spacy.load("en_ner_bc5cdr_md")

# NLP Function Extract Terms
def extract_medical_keywords(title):
    doc = nlp(title)
    medical_terms = [ent.text for ent in doc.ents if ent.label_ == "DISEASE"]
    return medical_terms

c:\Users\Thong Vo\anaconda3\envs\TfGPUPy310\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


# Medline domain

In [ ]:
# Select Medline
new_medline = []
for item in new_data["medline"]:
    if (("Summary" in item["content"]) or ("article" in item["url"])) and ("spanish" not in item["url"]) and ('magazine' not in item["url"]) and ('developer' not in item["url"]) and ('patientinstructions' not in item["url"]) and ('lab-tests' not in item["url"]) and ('allofus' not in item["url"]):
        new_medline.append(item)

# Clean Medline
for item in new_medline:
    if ("article" not in item["url"]):

        end_pos = len(item["content"])
        sh_pos = [i.start() for i in re.finditer("Start Here", item["content"])] 
        if (len(sh_pos) != 0):
            end_pos = min(end_pos, sh_pos[-1])
        else:
            lm_pos = [i.start() for i in re.finditer("Learn More", item["content"])] 
            if (len(lm_pos) != 0 ):
                end_pos = min(end_pos, lm_pos[-1])
            else:
                dat_pos = [i.start() for i in re.finditer("Diagnosis and Tests", item["content"])] 
                if (len(dat_pos) != 0):
                    end_pos = min(end_pos, dat_pos[-1])
        item["content"] = item["content"][:end_pos]

        start_pos_ar = [i.start() for i in re.finditer("Summary", item["content"])] 
        start_pos = start_pos_ar[-1]
        item["content"] = item["content"][start_pos+8:]

    else:
        start_pos_ar = [i.start() for i in re.finditer("To use the sharing features on this page, please enable JavaScript.", item["content"])] 
        start_pos = start_pos_ar[-1]
        item["content"] = item["content"][start_pos+68:]

        end_pos_ar = [i.start() for i in re.finditer("References", item["content"])]
        end_pos = end_pos_ar[-1]
        item["content"] = item["content"][:end_pos]  



In [5]:
# Extract Term in Medline title
for item in new_medline:
    if ("article" not in item["url"]):
        pos_ar = []
        if (":" in item["title"]):
            pos_ar = [i.start() for i in re.finditer(":", item["title"])]    
        else:
            pos_ar = [i.start() for i in re.finditer(r"\|", item["title"])]
        if (len(pos_ar) == 0):
            pos = len(item["title"])
        else:
            pos = pos_ar[0]
        item["term"] = item["title"][:pos].strip().lower()
    else:
        if ("-" in item["title"]):
            pos_ar = [i.start() for i in re.finditer("-", item["title"])]
        else:
            pos_ar = [i.start() for i in re.finditer(":", item["title"])]
        pos = pos_ar[0]
        item["term"] = item["title"][:pos].strip().lower()


# Medscape domain

In [ ]:
# Select Mescape
new_medscape = []
for item in new_data['medscape']:
    if ("overview" in item["url"]):
        new_medscape.append(item)

# Clean Mescape
for item in new_medscape:
    ref_pos_ar = [i.start() for i in re.finditer("References", item["content"])]
    end_pos = ref_pos_ar[1]
    item["content"] = item["content"][:end_pos]

    ov_pos_ar = [i.start() for i in re.finditer("Overview", item["content"])]        

    start_pos = ref_pos_ar[0]
    item["content"] = item["content"][start_pos+11:]

In [ ]:
# Extract Term in Mescape title
for item in new_medscape:
    title = item["title"]
    title = re.sub(':', '', title)
    terms = extract_medical_keywords(title)
    if (len(terms) != 0):
        item["term"] = terms[0].lower()
    else:
        item["term"] = ""

for item in new_medscape:
    if (len(item["term"]) == 0):
        new_medscape.remove(item)

# WebMD domain

In [ ]:
# Select WebMD
new_webmd = []
for item in new_data["webmd"]:
    if (("/heart-disease/" in item["url"]) or ("hypertension" in item["url"])) and ("default" not in item["url"]) and ("min read" in item["content"]):
        new_webmd.append(item)

# Clean WebMD
for item in new_webmd:
    mr_pos_ar = [i.start() for i in re.finditer("min read", item["content"])]
    start_pos = mr_pos_ar[0]
    item["content"] = item["content"][start_pos+9:]

    sources_pos_ar = [i.start() for i in re.finditer("Sources", item["content"])]
    end_pos = sources_pos_ar[0]
    item["content"] = item["content"][:end_pos] 

In [ ]:
# Extract Term in WebMD title
for item in new_webmd:
    title = item["title"]
    title = re.sub(':', '', title)
    terms = extract_medical_keywords(title)
    if (len(terms) != 0):
        item["term"] = terms[0].lower()
    else:
        if ("heart attack" in item["title"].lower()):
            item["term"] = "heart attack"
        else:
            if ("high blood pressure" in item["title"].lower()):
                item["term"] = "high blood pressure"
            else:
                item["term"] = ""

for item in new_webmd:
    if (len(item["term"]) == 0):
        new_webmd.remove(item)

# Save data to shortcut database

In [ ]:
# Add all together and save

shortcut_data = list(chain(new_medline, new_medscape, new_webmd))

with open('shortcut_database.json', 'w') as fo:
    json.dump(shortcut_data, fo)